# 15 (Kaggle) — Bảng 2: ablation modality C, D, C+M, D+M cho đủ 5 fold

Bảng 2 của bài ACIIDS hiện chỉ có **fold 0** cho 4 variant chỉ một ảnh. Notebook này chạy **fold 1–4** còn thiếu, đúng
cấu hình stage 6 (`experiments/stage6_ablation.yaml`): Swin-B-384 · MoE-D · `d_model 256` · `r5_nobce` · 3 head long-tail ·
batch 8 × 8 = 64 · grad checkpointing. Fold 0 đã có (val: C 0.491 · D 0.567 · C+M 0.503 · D+M 0.565) nên không chạy lại.

| Variant | Ảnh | Metadata | Run |
|---|---|---|---|
| `C` | clinical | không | `swin_base_384__abl_C__f{1..4}` |
| `D` | dermoscopic | không | `swin_base_384__abl_D__f{1..4}` |
| `CM` | clinical | có | `swin_base_384__abl_CM__f{1..4}` |
| `DM` | dermoscopic | có | `swin_base_384__abl_DM__f{1..4}` |

**Tổng 16 run.** Run một ảnh nhẹ hơn M0 (1 ảnh/lesion) nhưng vẫn ~1–1.5 giờ/run trên T4 (ước lượng) → 16 run trên 2 GPU
**~9–12 giờ**, sát giới hạn 12 giờ của Kaggle. Vì vậy chia **2 phiên**:
1. **Phiên 1:** `FOLDS = [1, 2]` (8 run, ~5–6 giờ) → *Save & Run All (Commit)* → tải zip.
2. **Phiên 2:** `FOLDS = [3, 4]`; gắn output phiên 1 làm Input và điền vào `RESTORE` để bảng cuối gộp đủ 4 fold
   (hoặc chạy phiên 2 song song trên tài khoản Kaggle khác, rồi gộp 2 zip ở máy).

`TIME_BUDGET_H` chặn không bắt đầu run mới nếu không kịp xong; run chưa chạy sẽ được chạy ở phiên sau (run đã xong bỏ qua).
Lưu ý: fold 0 chạy trên A100 (bf16), fold 1–4 chạy trên T4 (fp16); cùng recipe và batch hiệu dụng.

**Cách chạy:** Settings → **GPU T4 x2**, **Internet: On** → **Save Version → Save & Run All (Commit)**; tải zip ở tab *Output*.

In [ ]:
# ===================== CẤU HÌNH =====================
REPO_URL = "https://github.com/trong5nhan6/fussion.git"
BRANCH = "main"
DRIVE_URL = "https://drive.google.com/file/d/1lM1nB_4pq3Vg-GwlaTMQjtus_bpbMhGX/view?usp=drive_link"  # zip dữ liệu (tải công khai bằng gdown)
DATA_SOURCE = None    # hoặc đường dẫn Kaggle Dataset, vd "/kaggle/input/milk10k" (khi đó không tải từ link)

# Phiên 2: gắn output phiên 1 làm Input rồi liệt kê thư mục outputs của nó (run đã xong được chép về và bỏ qua)
RESTORE = []
TIME_BUDGET_H = 11.0

# ===================== CHỌN THÍ NGHIỆM =====================
EXPERIMENT = "experiments/stage6_ablation.yaml"
BACKBONES = ["swin_base_384"]
VARIANTS = ["C", "D", "CM", "DM"]   # ảnh đơn, có/không metadata (Bảng 2)
FOLDS = [1, 2]                      # phiên 1: [1, 2]  |  phiên 2: [3, 4]

In [ ]:
# ===================== THAM SỐ TRAIN (giống stage 6) =====================
# Batch / grad checkpointing / d_model / lr đã đặt trong stage6_ablation.yaml + overlay r5_nobce. T4 không có bf16 -> fp16.
PARAMS = {
    "train.amp_dtype": "fp16",
    "train.epochs": 20,
    "train.patience": 5,
    "seed": 42,
    "predict.tta": True,
}
RUN_SUFFIX = ""

## 1. Cài đặt

In [ ]:
import os, glob
from IPython.display import Markdown, display

REPO_DIR = "/kaggle/working/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
DATA_DIR = "/tmp/MILK10k"               # ngoài /kaggle/working -> dữ liệu không bị lưu vào output notebook
os.environ["MILK10K_ROOT"] = DATA_DIR   # code tự đọc biến này thay cho data.root trong config

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q gdown
!git log -1 --format="Commit: %h %s"

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| số GPU:", torch.cuda.device_count())
assert torch.cuda.device_count() >= 1, "Chưa bật GPU: Settings -> Accelerator -> GPU T4 x2"

## 2. Dữ liệu

In [ ]:
src = f"--source {DATA_SOURCE}" if DATA_SOURCE else f'--drive-url "{DRIVE_URL}"'
!python scripts/setup_data.py {src} --dest {DATA_DIR}

In [ ]:
# Kiểm tra nhanh pipeline MoE (~1 phút, không tải trọng số)
!python -m pytest tests/test_smoke.py -q -p no:cacheprovider -k "moe"

## 3. Danh sách run và cấu hình thực tế

In [ ]:
restore_arg = ("--restore " + " ".join(RESTORE)) if RESTORE else ""
for r in RESTORE:
    print(r, "->", "OK" if os.path.isdir(r) else "KHÔNG TỒN TẠI (kiểm tra lại đường dẫn Input)")

In [ ]:
import shlex, yaml

def to_set_args(params: dict) -> str:
    # Chuyển PARAMS thành chuỗi --set key=value cho scripts (bỏ qua giá trị None)
    items = []
    for k, v in params.items():
        if v is None:
            continue
        val = yaml.safe_dump(v, default_flow_style=True).strip().removesuffix("...").strip()
        items.append(shlex.quote(f"{k}={val}"))
    return ("--set " + " ".join(items)) if items else ""

set_arg = to_set_args(PARAMS)
suffix_arg = f"--suffix {RUN_SUFFIX}" if RUN_SUFFIX else ""
print("Override:", set_arg or "(không có)")
print("Hậu tố tên run:", RUN_SUFFIX or "(không có)")

In [ ]:
select_arg = (f"--backbones {','.join(BACKBONES)} --variants {','.join(VARIANTS)}"
              + (f" --folds {','.join(map(str, FOLDS))}" if FOLDS else ""))
n_runs = len(VARIANTS) * len(FOLDS)
print(f"{n_runs} run, ước tính ~{75 * n_runs / 2 / 60:.1f} giờ trên 2 GPU (giới hạn phiên: {TIME_BUDGET_H} giờ)")
!python scripts/run_grid.py {EXPERIMENT} --env configs/env/kaggle_t4.yaml {select_arg} {set_arg} {suffix_arg} {restore_arg} --dry-run

## 4. Train

In [ ]:
!python scripts/run_grid.py {EXPERIMENT} --env configs/env/kaggle_t4.yaml {select_arg} {set_arg} {suffix_arg} {restore_arg} --gpus auto --est-run-h 1.3 --time-budget-h {TIME_BUDGET_H}

## 5. Bảng 2: Dice val theo fold (fold 0 lấy từ stage 6)
Mỗi ô là Dice val của một fold; cột cuối là mean ± std trên các fold đã có. Bảng lưu `outputs/table2_modality.csv`.

In [ ]:
import json, glob, os
import numpy as np
import pandas as pd
FOLD0 = {"C": 0.491, "D": 0.567, "CM": 0.503, "DM": 0.565}          # stage 6 (A100), không chạy lại
FIVE = {"C+D": (0.575, 0.014), "C+D+M": (0.584, 0.014)}             # đã có 5 fold (stage 6)
dice = {v: {0: d} for v, d in FOLD0.items()}
for d in glob.glob("outputs/swin_base_384__abl_*__f*"):
    name = d.rstrip("/").split("/")[-1]
    v, f = name.split("__abl_")[1].split("__f")
    if v in dice and os.path.exists(f"{d}/metrics.json"):
        dice[v][int(f)] = json.load(open(f"{d}/metrics.json"))["overall"]["dice"]
rows = []
for v, per in dice.items():
    vals = np.array([per[k] for k in sorted(per)])
    rows.append({"variant": v, **{f"fold {k}": per[k] for k in sorted(per)}, "số fold": len(vals),
                 "mean ± std": f"{vals.mean():.3f} ± {vals.std(ddof=1):.3f}" if len(vals) > 1 else f"{vals[0]:.3f}"})
tab = pd.DataFrame(rows).set_index("variant")
display(tab.round(3))
print("Đã có 5 fold từ trước:", {k: f"{m:.3f} ± {s:.3f}" for k, (m, s) in FIVE.items()})
tab.to_csv("outputs/table2_modality.csv")

## Nén kết quả để tải về

In [ ]:
import zipfile, datetime
from pathlib import Path

zip_path = Path("/kaggle/working") / f"15_modality_folds_kaggle_outputs_{datetime.datetime.now():%Y%m%d_%H%M}.zip"
src = Path(REPO_DIR) / "outputs"
n = 0
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(src.rglob("*")):
        if f.is_file() and f.suffix != ".pt":          # không đóng gói checkpoint
            z.write(f, Path("outputs") / f.relative_to(src))
            n += 1
print(f"Đã nén {n} file -> {zip_path} ({zip_path.stat().st_size / 2**20:.1f} MB)")
print("Tải về: tab Output của version notebook (bên phải) -> {} -> Download".format(zip_path.name))